In [1]:
# Install any missing libraries
import subprocess
subprocess.run(["pip", "install", "scikit-learn", "pandas", "boto3", "joblib"], capture_output=True)

CompletedProcess(args=['pip', 'install', 'scikit-learn', 'pandas', 'boto3', 'joblib'], returncode=0, stdout=b'Requirement already satisfied: scikit-learn in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (1.8.0)\nRequirement already satisfied: pandas in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (2.3.3)\nRequirement already satisfied: boto3 in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (1.42.84)\nRequirement already satisfied: joblib in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (1.5.3)\nRequirement already satisfied: numpy>=1.24.1 in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (from scikit-learn) (1.26.4)\nRequirement already satisfied: scipy>=1.10.0 in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (from scikit-learn) (1.17.1)\nRequirement already satisfied: threadpoolctl>=3.2.0 in /home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages (fr

In [2]:
import boto3
import pandas as pd
from io import StringIO

# Download processed CSVs from S3
s3 = boto3.client('s3', region_name='us-east-2')
bucket = 'aws-ids-platform'
prefix = 'processed/friday-ddos-clean/'

# List all part files
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
files = [obj['Key'] for obj in response['Contents'] if obj['Key'].endswith('.csv')]

# Load and combine all parts
dfs = []
for f in files:
    obj = s3.get_object(Bucket=bucket, Key=f)
    dfs.append(pd.read_csv(obj['Body']))

df = pd.concat(dfs, ignore_index=True)
print(f"Dataset shape: {df.shape}")
print(f"Label distribution:\n{df['Label'].value_counts()}")

Dataset shape: (225741, 79)
Label distribution:
Label
DDoS      128027
BENIGN     97714
Name: count, dtype: int64


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import numpy as np

# Prepare features and labels
X = df.drop(columns=['Label'])
y = (df['Label'] == 'DDoS').astype(int)  # 1 = attack, 0 = normal

# Replace infinities with NaN then drop
X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna(axis=1)  # drop columns with any NaN
y = y[X.index]        # keep labels aligned

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {X_train.shape[0]} samples with {X_train.shape[1]} features")

# Train model
model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

# Evaluate
print("\nClassification Report:")
print(classification_report(y_test, model.predict(X_test)))

Training on 180592 samples with 76 features

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19417
           1       1.00      1.00      1.00     25732

    accuracy                           1.00     45149
   macro avg       1.00      1.00      1.00     45149
weighted avg       1.00      1.00      1.00     45149



In [4]:
import joblib
import os

# Save model locally in the notebook
joblib.dump(model, '/tmp/rf_model.joblib')
print("Model saved locally")

# Upload to S3
s3.upload_file('/tmp/rf_model.joblib', 'aws-ids-platform', 'models/rf_model.joblib')
print("Model uploaded to s3://aws-ids-platform/models/rf_model.joblib")

# Also save the feature column names — needed later for Lambda inference
import json
feature_cols = list(X.columns)
with open('/tmp/feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)

s3.upload_file('/tmp/feature_cols.json', 'aws-ids-platform', 'models/feature_cols.json')
print(f"Feature columns saved ({len(feature_cols)} features)")

Model saved locally
Model uploaded to s3://aws-ids-platform/models/rf_model.joblib
Feature columns saved (76 features)


In [5]:
import sklearn
print(sklearn.__version__)

1.8.0


In [6]:
import subprocess
subprocess.run(["pip", "install", "scikit-learn==1.7.2"], capture_output=True)

import importlib, sklearn
importlib.reload(sklearn)

# Retrain and resave with 1.7.2
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)

import joblib
joblib.dump(model, '/tmp/rf_model.joblib')
s3.upload_file('/tmp/rf_model.joblib', 'aws-ids-platform', 'models/rf_model.joblib')
print("Model resaved with sklearn 1.7.2")

Model resaved with sklearn 1.7.2


In [7]:
# Get a real DDoS row to use as test
ddos_row = df[df['Label'] == 'DDoS'].iloc[0]
test_payload = ddos_row[feature_cols].to_dict()
test_payload['event_id'] = 'test-real-ddos-001'
test_payload['timestamp'] = '2026-05-01T02:00:00Z'

import json
print(json.dumps(test_payload))

{"Destination Port": 80.0, "Flow Duration": 1293792.0, "Total Fwd Packets": 3.0, "Total Backward Packets": 7.0, "Total Length of Fwd Packets": 26.0, "Total Length of Bwd Packets": 11607.0, "Fwd Packet Length Max": 20.0, "Fwd Packet Length Min": 0.0, "Fwd Packet Length Mean": 8.666667, "Fwd Packet Length Std": 10.263203, "Bwd Packet Length Max": 5840.0, "Bwd Packet Length Min": 0.0, "Bwd Packet Length Mean": 1658.1428, "Bwd Packet Length Std": 2137.297, "Flow IAT Mean": 143754.67, "Flow IAT Std": 430865.8, "Flow IAT Max": 1292730.0, "Flow IAT Min": 2.0, "Fwd IAT Total": 747.0, "Fwd IAT Mean": 373.5, "Fwd IAT Std": 523.9661, "Fwd IAT Max": 744.0, "Fwd IAT Min": 3.0, "Bwd IAT Total": 1293746.0, "Bwd IAT Mean": 215624.33, "Bwd IAT Std": 527671.94, "Bwd IAT Max": 1292730.0, "Bwd IAT Min": 2.0, "Fwd PSH Flags": 0.0, "Bwd PSH Flags": 0.0, "Fwd URG Flags": 0.0, "Bwd URG Flags": 0.0, "Fwd Header Length34": 72.0, "Bwd Header Length": 152.0, "Fwd Packets/s": 2.3187654, "Bwd Packets/s": 5.4104524,